# Huấn luyện mô hình nhận diện khuôn mặt

Notebook này dùng để huấn luyện mô hình nhận diện khuôn mặt từ dataset đã thu thập.

## 1. Cài đặt thư viện

In [ ]:
# Cài đặt các thư viện cần thiết
!pip install opencv-python
!pip install face-recognition
!pip install scikit-learn
!pip install numpy
!pip install pillow

## 2. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Đường dẫn đến thư mục dự án
PROJECT_PATH = '/content/drive/MyDrive/face_attendance'

import os
os.chdir(PROJECT_PATH)

## 3. Import thư viện

In [ ]:
import cv2
import os
import pickle
import numpy as np
import face_recognition
from sklearn.svm import SVC
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report

## 4. Cấu hình

In [ ]:
# Cấu hình
DATASET_DIR = 'dataset'
MODEL_DIR = 'models'
MODEL_PATH = os.path.join(MODEL_DIR, 'face_recognition_model.pkl')
FACE_ENCODING_MODEL = 'large'  # 'small' hoặc 'large'

# Tạo thư mục models nếu chưa có
os.makedirs(MODEL_DIR, exist_ok=True)

## 5. Load dataset và tạo encodings

In [ ]:
def load_dataset():
    """
    Load dataset và tạo face encodings
    """
    encodings = []
    names = []
    
    # Lấy danh sách người
    persons = [d for d in os.listdir(DATASET_DIR) 
               if os.path.isdir(os.path.join(DATASET_DIR, d))]
    
    print(f"Tìm thấy {len(persons)} người trong dataset")
    
    total_images = 0
    
    for person_name in persons:
        person_dir = os.path.join(DATASET_DIR, person_name)
        image_files = [f for f in os.listdir(person_dir) 
                      if f.endswith(('.jpg', '.jpeg', '.png'))]
        
        print(f"\nĐang xử lý {person_name}: {len(image_files)} ảnh")
        
        for i, img_file in enumerate(image_files):
            img_path = os.path.join(person_dir, img_file)
            
            try:
                # Đọc ảnh
                image = cv2.imread(img_path)
                if image is None:
                    print(f"  ✗ Không thể đọc: {img_file}")
                    continue
                
                # Chuyển sang RGB
                rgb_image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
                
                # Tạo encoding
                face_encodings = face_recognition.face_encodings(
                    rgb_image,
                    model=FACE_ENCODING_MODEL
                )
                
                if len(face_encodings) > 0:
                    encodings.append(face_encodings[0])
                    names.append(person_name)
                    total_images += 1
                    
                    if (i + 1) % 10 == 0:
                        print(f"  Đã xử lý {i + 1}/{len(image_files)} ảnh")
                else:
                    print(f"  ✗ Không phát hiện khuôn mặt: {img_file}")
            
            except Exception as e:
                print(f"  ✗ Lỗi khi xử lý {img_file}: {str(e)}")
        
        print(f"  ✓ Hoàn thành {person_name}")
    
    print(f"\n✓ Tổng cộng đã tạo encoding cho {total_images} ảnh")
    
    return encodings, names

# Load dataset
print("Bắt đầu load dataset...")
encodings, names = load_dataset()

## 6. Chuẩn bị dữ liệu

In [ ]:
# Chuyển sang numpy array
X = np.array(encodings)
y = np.array(names)

print(f"Shape của X: {X.shape}")
print(f"Shape của y: {y.shape}")
print(f"Số lượng người: {len(np.unique(y))}")
print(f"Danh sách người: {np.unique(y)}")

## 7. Encode labels

In [ ]:
# Encode labels
label_encoder = LabelEncoder()
y_encoded = label_encoder.fit_transform(y)

print(f"Classes: {label_encoder.classes_}")
print(f"Encoded labels: {np.unique(y_encoded)}")

## 8. Chia train/test

In [ ]:
# Chia train/test (80/20)
X_train, X_test, y_train, y_test = train_test_split(
    X, y_encoded, test_size=0.2, random_state=42, stratify=y_encoded
)

print(f"Training set: {X_train.shape[0]} samples")
print(f"Test set: {X_test.shape[0]} samples")

## 9. Huấn luyện mô hình SVM

In [ ]:
# Huấn luyện SVM
print("Bắt đầu huấn luyện mô hình SVM...")

model = SVC(
    kernel='linear',
    probability=True,
    C=1.0,
    gamma='scale',
    verbose=True
)

model.fit(X_train, y_train)

print("\n✓ Hoàn thành huấn luyện!")

## 10. Đánh giá mô hình

In [ ]:
# Dự đoán trên tập test
y_pred = model.predict(X_test)

# Tính accuracy
accuracy = accuracy_score(y_test, y_pred)
print(f"Accuracy: {accuracy * 100:.2f}%")

# Classification report
print("\nClassification Report:")
print(classification_report(
    y_test, 
    y_pred, 
    target_names=label_encoder.classes_
))

## 11. Lưu mô hình

In [ ]:
# Lưu mô hình và các thông tin liên quan
data = {
    'model': model,
    'label_encoder': label_encoder,
    'encodings': encodings,
    'names': names,
    'accuracy': accuracy
}

with open(MODEL_PATH, 'wb') as f:
    pickle.dump(data, f)

print(f"✓ Đã lưu mô hình vào {MODEL_PATH}")
print(f"Accuracy: {accuracy * 100:.2f}%")
print(f"Số lượng người: {len(label_encoder.classes_)}")

## 12. Test mô hình với ảnh mẫu

In [ ]:
# Test với một ảnh ngẫu nhiên
import random
from google.colab.patches import cv2_imshow

# Chọn ngẫu nhiên một người và một ảnh
test_person = random.choice(label_encoder.classes_)
test_person_dir = os.path.join(DATASET_DIR, test_person)
test_images = [f for f in os.listdir(test_person_dir) if f.endswith('.jpg')]
test_image_path = os.path.join(test_person_dir, random.choice(test_images))

print(f"Test với ảnh của: {test_person}")
print(f"Đường dẫn: {test_image_path}")

# Đọc và xử lý ảnh
test_img = cv2.imread(test_image_path)
test_rgb = cv2.cvtColor(test_img, cv2.COLOR_BGR2RGB)

# Tạo encoding
test_encoding = face_recognition.face_encodings(test_rgb, model=FACE_ENCODING_MODEL)

if len(test_encoding) > 0:
    # Dự đoán
    prediction = model.predict([test_encoding[0]])[0]
    predicted_name = label_encoder.inverse_transform([prediction])[0]
    
    # Lấy xác suất
    probabilities = model.predict_proba([test_encoding[0]])[0]
    confidence = probabilities[prediction]
    
    print(f"\nDự đoán: {predicted_name}")
    print(f"Độ tin cậy: {confidence * 100:.2f}%")
    print(f"Kết quả: {'✓ ĐÚNG' if predicted_name == test_person else '✗ SAI'}")
    
    # Hiển thị ảnh
    cv2_imshow(test_img)
else:
    print("Không phát hiện khuôn mặt trong ảnh test")